## CSCE 676 :: Data Mining and Analysis :: Texas A&M University :: Fall 2025


# Homework 1: Let's GOOOOO!

- **100 points [7.5% of your final grade]**
- **Due Tuesday, September 14 by 11:59pm**

***Goals of this homework:***
1. Collect data from the web, clean it, and then make some observations based on exploratory data analysis
2. Understand and implement the classic apriori algorithm and extensions to find the association rules in a movie rating dataset
3. Push yourself by using Spark to find association rules

***Submission instructions:***

You should post your notebook to Canvas (look for the homework 1 assignment there). Please name your submission **your-uin_hw1.ipynb**, so for example, my submission would be something like **555001234_hw1.ipynb**. Your notebook should be fully executed when you submit ... so run all the cells for us so we can see the output, then submit that.

***Late Days:***

As a reminder, you begin the semester with five late days. You may use as many as you like. There is no need to alert us to how many late days you are using. Just submit and we will make note of it. Also remember that once your late days are used up, homeworks will receive a 0.

***Collaboration and AI Assistance declaration:***

If you worked with someone on this homework, please be sure to mention that. Remember to include citations to any sources you use in the homework. Also tell us what AI assistant you used and how you used it.

## (REQUIRED) Collaboration and AI Assistance Declaration

### Collaboration Declaration:

*your response goes here*

### AI Assistance Declaration:
gpt4.1 vscode chat for helper functions or quick syntax on longer operations
haversine for lat/long->mi


## (45 points) Part 1: UFO Sightings — Data Ingestion, Cleaning, and Feature Engineering

**Dataset:** `ufos.csv`

Detected columns: `datetime`, `city`, `state`, `country`, `shape`, `duration (seconds)`, `duration (hours/min)`, `comments`, `date posted`, `latitude`, `longitude`, and possibly extra unnamed columns.

**Goal:** Load the data, diagnose issues, clean/standardize it, and derive basic features to support downstream mining. You will probably want to use `pandas` for this.



### (5pts) Part 1a: Load and sanity-check the raw data

Tasks:
1. Load `ufos.csv` into a DataFrame named `ufo_raw`.
2. Display 5 random rows and `ufo_raw.info()`.
3. Briefly report the number of rows/columns and any obviously empty columns.


In [2]:
import pandas as pd
pd.set_option('display.max_rows', None)

# Read columns as string first
ufo_raw = pd.read_csv(
    'ufos.csv',
    dtype=str,
    parse_dates=['datetime', 'date posted'],
    on_bad_lines='skip'
)

# Display 5 random rows and info
from random import randint
display(ufo_raw.sample(5, random_state=randint(0, 10000)))
ufo_raw.info()

def display_info(df):
    df.info()
    display(df.describe(include='all'))

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
34143,3/12/1997 19:20,hotedrsica (slovenia),NaN,NaN,light,6,6 sec,Formation of approx. 30 glowing objects seen i...,937094400000000000,45.930065,14.139957
77808,8/29/2004 22:30,london (uk/england),NaN,gb,sphere,30,30 seconds,We were on a main road in London called Finchl...,1093996800000000000,51.514125,-.093689
55728,6/20/1974 05:30,ashland,oh,us,disk,60,1 minute,Low flying disc shaped object with colored lig...,1177632000000000000,40.8686111,-82.3183333
21715,12/2/2004 21:00,del rio,tx,us,other,180,3 min,On Dec 2nd two strange lights appeared around ...,1102982400000000000,29.3625000,-100.8963889
20783,12/18/1999 22:30,groveland/dillion (between),il,NaN,triangle,900,15 mins,THREE BRIGHT LIGHTS HOVERING ABOVE. BOYFRIEND ...,948499200000000000,40.633125,-89.398528


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88679 entries, 0 to 88678
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   datetime              88679 non-null  object
 1   city                  88679 non-null  object
 2   state                 81270 non-null  object
 3   country               76314 non-null  object
 4   shape                 85757 non-null  object
 5   duration (seconds)    88677 non-null  object
 6   duration (hours/min)  85660 non-null  object
 7   comments              88644 non-null  object
 8   date posted           88679 non-null  object
 9   latitude              88679 non-null  object
 10  longitude             88679 non-null  object
dtypes: object(11)
memory usage: 7.4+ MB


`datetime:`, `city:str`, `state`, `country`, `shape`, `duration (seconds)`, `duration (hours/min)`, `comments`, `date posted`, `latitude`, `longitude`


### (5pts) Part 1b: Clean up datatypes & columns

Create a cleaned DataFrame `ufo`:

- Drop fully-empty or irrelevant columns (e.g., unnamed columns).
- Parse `datetime` to `datetime64[ns]` (`errors='coerce'`).
- Coerce `duration (seconds)`, `latitude`, `longitude` to numeric.
- Lowercase/trim `city`, `state`, `country`, `shape`.
- Remove rows with impossible coordinates (lat ∉ [-90,90], lon ∉ [-180,180]).
- Drop exact duplicates based on a reasonable subset (document your choice).

Provide a short markdown note explaining your choices.


In [3]:
import re
from datetime import datetime, timedelta

# datetime values with 24:00 get dropped when using pd.to_datetime
def fix_24_hour(dt_str):
    # Match date and time with 24:00
    match = re.match(r'(\d{1,2}/\d{1,2}/\d{2,4}) 24:00', str(dt_str))
    if match:
        # Parse date, add one day, set time to 00:00
        date_part = match.group(1)
        try:
            date_obj = datetime.strptime(date_part, '%m/%d/%Y')
        except ValueError:
            date_obj = datetime.strptime(date_part, '%m/%d/%y')
        new_dt = date_obj + timedelta(days=1)
        return new_dt.strftime('%m/%d/%Y 00:00')
    return dt_str

ufo = pd.DataFrame(ufo_raw)
# make 24:00 datetimes readable to pandas
ufo['datetime'] = ufo['datetime'].apply(fix_24_hour)
ufo['datetime'] = pd.to_datetime(ufo['datetime'], errors='coerce')

# Convert numeric columns, coercing errors to NaN
for col in ['duration (seconds)', 'latitude', 'longitude']:
    ufo[col] = pd.to_numeric(ufo[col], errors='coerce')
ufo = ufo[
    (ufo['latitude'] >= -90) & (ufo['latitude'] <= 90) &
    (ufo['longitude'] >= -180) & (ufo['longitude'] <= 180)
]
# Replace latitude and longitude values of 0 with NaN
ufo.loc[ufo['latitude'] == 0, 'latitude'] = None
ufo.loc[ufo['longitude'] == 0, 'longitude'] = None


# Convert other columns to desired types if needed
for col in ['city', 'state', 'country', 'shape', 'duration (hours/min)', 'comments']:
    ufo[col] = ufo[col].astype(str)
# Replace 'yk' with 'yt' in the 'state' column for the more commonly abv yukon territory
ufo['state'] = ufo['state'].replace('yk', 'yt')
# Set 'state' to 'vi' and 'pr' and 'us' for virgin islands and puerto rico
for x, y in [('virgin islands', 'vi'), ('puerto rico', 'pr')]:
    ufo.loc[
        (ufo['city'].str.lower().str.contains(x)) &
        ((ufo['state'].str.strip() == '') | (ufo['state'].str.lower() == 'nan')),
        'state'
    ] = y
    ufo.loc[
        (ufo['city'].str.lower().str.contains(x)) &
        ((ufo['country'].str.strip() == '') | (ufo['country'].str.lower() == 'nan')),
        'country'
    ] = 'us'
# for the more common country codes left blank
for x, y in [('canada', 'ca'), ('new zealand', 'nz'), ('south africa', 'za'), ('singapore', 'sg'), ('ireland', 'ie'), ('(italy)', 'it'), ('(mexico)', 'mx'), ('(uk/england)', 'gb'), ('(russia)', 'ru'), ('(lithuania)', 'lt'), ('(romania)', 'ro'), ('(india)', 'in'), ('iran', 'ir'), ('(iraq)', 'iq'), ('malaysia', 'myq'), ('australia', 'au'),('china', 'cn'), ('spain', 'es'), ('poland', 'pl'), ('greece', 'gr'), ('brazil', 'br'), ('portugal', 'pt'), ('south korea', 'kr'), ('japan', 'jp'), ('philippines', 'ph')]:
    ufo.loc[ # regex false because I want to include some parenthesis in city field
        (ufo['city'].str.lower().str.contains(x, regex=False)) &
        ((ufo['country'].str.strip() == '') | (ufo['country'].str.lower() == 'nan')),
        'country'
    ] = y
# List of two-letter US state abbreviations
us_states = [
    'al', 'ak', 'az', 'ar', 'ca', 'co', 'ct', 'de', 'fl', 'ga', 'hi', 'id', 'il', 'in', 'ia', 'ks', 'ky', 'la', 'me',
    'md', 'ma', 'mi', 'mn', 'ms', 'mo', 'mt', 'ne', 'nv', 'nh', 'nj', 'nm', 'ny', 'nc', 'nd', 'oh', 'ok', 'or', 'pa',
    'ri', 'sc', 'sd', 'tn', 'tx', 'ut', 'vt', 'va', 'wa', 'wv', 'wi', 'wy', 'dc'
]
# Set country to 'us' if state is a US abbreviation and country is missing or NaN
ufo.loc[
    (ufo['state'].str.lower().isin(us_states)) & (ufo['country'] == 'nan'),
    'country'
] = 'us'

ufo.info()
# duplicates = ufo[ufo.duplicated(keep=False)]
# print(duplicates)
# > no duplicates found

<class 'pandas.core.frame.DataFrame'>
Index: 88678 entries, 0 to 88678
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              88678 non-null  datetime64[ns]
 1   city                  88678 non-null  object        
 2   state                 88678 non-null  object        
 3   country               88678 non-null  object        
 4   shape                 88678 non-null  object        
 5   duration (seconds)    88676 non-null  float64       
 6   duration (hours/min)  88678 non-null  object        
 7   comments              88678 non-null  object        
 8   date posted           88678 non-null  object        
 9   latitude              87184 non-null  float64       
 10  longitude             87184 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(7)
memory usage: 8.1+ MB


### (5pts) Part 1c: Be thankful!

Now take a look at the `duration (hours/min)` column. In a previous version of this homework, students spent upwards of 10-20 hours just cleaning the values in this column. Luckily for you, the rest of this assignment relies on the much nicer `duration (seconds)` column. For this question, we'd like you to just extract how ever many versions of durations reported in *minutes* you can from the `duration (hours/min)` column. In other words, find as many different variations of anything that could be reasonably interpreted as a minute-like duration. Examples include:

* several minutes
* x to y minutes
* x minutes
* x min.
* x mins.
* and so on ...


In [4]:
print(ufo['duration (hours/min)'].value_counts())
print(f'{len(ufo['duration (hours/min)'].value_counts())} different variations (including nan)')

duration (hours/min)
5 minutes                             4796
2 minutes                             3538
10 minutes                            3383
1 minute                              3118
nan                                   3019
3 minutes                             2553
30 seconds                            2350
15 minutes                            2105
10 seconds                            2016
5 seconds                             1836
20 minutes                            1467
1 hour                                1353
30 minutes                            1353
15 seconds                            1194
20 seconds                            1149
3 seconds                              945
4 minutes                              918
5 min                                  858
2 seconds                              699
10 min                                 662
2 hours                                661
2-3 minutes                            619
2 min                            

In [5]:
# Replace any sequence of digits with 'x' in the 'duration (hours/min)' column and create a new Series
duration_x = ufo['duration (hours/min)'].str.replace(r'\d+', 'x', regex=True)
print(duration_x.value_counts())
print(f'{len(duration_x.value_counts())} different variations when we remove specific numbers')
# If I were to try to clean this column I would pull the first number, scan for 'm','h', then 's' and set the value to num,num*60,num/60 respectively
# this would lose accuracy for fractioned or interval entries, but would give us something close to work with

duration (hours/min)
x minutes                          22776
x seconds                          13190
x min                               4779
x-x minutes                         3695
x minute                            3199
nan                                 3019
x-x seconds                         2404
x min.                              1835
x mins                              1787
x sec                               1601
x hour                              1360
x hours                             1227
x sec.                               987
xmin                                 972
x:x                                  885
x                                    683
x-x min                              592
unknown                              528
seconds                              525
x second                             410
x secs                               385
x to x minutes                       362
xminutes                             350
x-x min.                            

In [6]:
# Only pull values from 'duration (hours/min)' that contain 'm', then replace digits with 'x'
duration_x_m = ufo.loc[ufo['duration (hours/min)'].str.contains('m', case=False, na=False), 'duration (hours/min)'].str.replace(r'\d+', 'x', regex=True)
print(duration_x_m.value_counts())
print(f'{len(duration_x_m.value_counts())} different variations when we only include values believed to be in minutes')
# this approach is simple, assuming any value with 'm' is in minutes
# this will neglect values without units such as x:x (containing multiple units)

duration (hours/min)
x minutes                          22776
x min                               4779
x-x minutes                         3695
x minute                            3199
x min.                              1835
x mins                              1787
xmin                                 972
x-x min                              592
x to x minutes                       362
xminutes                             350
x-x min.                             317
x-x mins                             312
xmins                                303
~x minutes                           263
about x minutes                      224
x mins.                              206
xmin.                                183
x.x minutes                          170
one minute                           169
x - x minutes                        159
x+ minutes                           146
several minutes                      124
x-xmin                               116
few minutes                         


### (10pts) Part 1d: Feature engineering and small exploratory data analysis

Create additional columns in a new feature engineering version of the dataset called `ufo_fe`:

- `year`, `month`, `hour` from `datetime`
- `duration_log10` = log10(duration_seconds + 1)
- `duration_bucket` = categorical bins for `duration (seconds)` (you may adjust edges)
- `us_only` = 1 if `country == 'us'` else 0

Then produce:
- Value counts of `shape` (top 10).
- A table of sightings by `year` (counts).
- A `state × shape` table (top 10 states by sample size).


In [7]:
from math import log10
ufo_fe = pd.DataFrame(ufo)
ufo_fe['year'] = ufo_fe['datetime'].dt.year
ufo_fe['month'] = ufo_fe['datetime'].dt.month
ufo_fe['hour'] = ufo_fe['datetime'].dt.hour
ufo_fe['duration_log10'] = ufo_fe['duration (seconds)'].apply(lambda x: log10(x + 1))
# Create duration_bucket column with categorical bins for 'duration (seconds)'
bins = [0, 10, 60, 300, 1800, 3600, 21600, 86400, float('inf')]
labels = ['<10s', '10s-1m', '1m-5m', '5m-30m', '30m-1h', '1h-6h', '6h-24h', '>24h']
ufo_fe['duration_bucket'] = pd.cut(ufo_fe['duration (seconds)'], bins=bins, labels=labels, right=False)
ufo_fe['us_only'] = ufo_fe['country'].apply(lambda x: 1 if x == 'us' else 0)

print(f'Top 10 shapes:')
print(ufo_fe['shape'].value_counts()[:10])
year_counts = ufo_fe['year'].value_counts().sort_index().to_frame(name='count')
print(f'\nCount of Sightings by year:')
display(year_counts)
top_states = ufo_fe[~ufo_fe['state'].str.lower().isin(['nan', '', 'none'])]['state'].value_counts().head(10).index
top_states_df = ufo_fe[ufo_fe['state'].isin(top_states)][['state', 'shape']]
top_states_df = top_states_df.reset_index(drop=True)
print(f'description about the (large) state x shape table:')
top_states_df.describe()
print(top_states,'\n',top_states_df.columns)
print(top_states_df.value_counts())


Top 10 shapes:
shape
light       17872
triangle     8489
circle       8453
fireball     6562
unknown      6319
other        6247
disk         6005
sphere       5755
oval         4119
nan          2922
Name: count, dtype: int64

Count of Sightings by year:


,count
year,
1906,1
1910,3
1914,1
1916,1
1917,1
1920,2
1925,1
1929,1
1930,2


description about the (large) state x shape table:
Index(['ca', 'wa', 'fl', 'tx', 'ny', 'az', 'il', 'pa', 'oh', 'mi'], dtype='object', name='state') 
 Index(['state', 'shape'], dtype='object')
state  shape    
ca     light        2105
wa     light         979
ca     circle        977
       triangle      939
fl     light         877
tx     light         807
ca     fireball      785
       disk          763
       other         731
       sphere        707
az     light         663
ca     unknown       663
ny     light         660
il     light         583
pa     light         545
oh     light         498
fl     fireball      481
ca     oval          454
mi     light         452
fl     circle        432
tx     triangle      410
fl     triangle      409
wa     nan           406
       fireball      397
       triangle      378
       circle        375
ny     circle        363
ca     nan           362
wa     unknown       356
il     triangle      348
tx     circle        345
ny     triangle


### (5pts) Part 1e: Observations and conclusions

In 3–6 sentences, summarize data quality and distributional patterns; reference at least one artifact from 1d.


users frequently left columns blank\
few sightings before 1995, and very few before 1950\
most sightings in the US\
much of the less common free response data (like shape) could be classified as more common labels \(round, circle, sphere|triangle,delta|light,fireball|unknown,other)


### (5pts) Part 1e: Next steps

Propose 2–3 concrete next steps (e.g., better deduplication with fuzzy text, geospatial clustering, normalization of duration text, timezone handling) to improve the quality of the data before moving on to some downstream tasks (you don't need to implement these).


I think our chosen buckets for duration(s) represent the data well, with the bulk of our data between 1m-30m (in two buckets) then 0s-1m (in two buckets).\
Combining labels for shapes and checking accuracy of longitude and latitude against location data would improve the quality of our data.\
uniform representation of time (implemented) was important as well as we list times as both 24:00 and 00:00 which pandas could not handle.


### (10 points) Part 1 f: Scalable & Structured Patterns

Now let's explore **network**, **geospatial**, and **time** patterns to dig a little deeper into our data.
Complete **any two** sub‑tasks below with **pandas**. For each you should provide your code, the ouput, plus some written analysis of what you find.



#### Sub-task 1. Co‑witness graph (space‑time co‑occurrence)
Bucket by geocell (`lat1=round(latitude,1)`, `lon1=round(longitude,1)`) and `datetime` floored to 15 minutes.  
Two reports in the same bucket are co‑witnessed. Build edges between reports in each bucket and summarize the graph (top buckets, approximate largest component).



#### Sub-task 2.  Geospatial hotspots
Aggregate by (`lat1`,`lon1`) and list **top‑20** cells. Optionally justify a normalization (per capita proxies or surface area) if you apply one.


In [8]:
ufo_loc = ufo_fe[['country', 'state', 'city', 'latitude', 'longitude']].copy()

import numpy as np

# Haversine distance in miles
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # Radius of Earth in miles
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def fill_lat_long_within_radius(group, max_miles=10):
    known = group.dropna(subset=['latitude', 'longitude'])
    missing = group[group['latitude'].isna() | group['longitude'].isna()]

    if len(known) == 0 or len(missing) == 0:
        return group  # Nothing to fill or no knowns

    # Compute pairwise distances between known points
    coords = known[['latitude', 'longitude']].to_numpy()
    too_far = False
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            dist = haversine(*coords[i], *coords[j])
            if dist > max_miles:
                too_far = True
                break
        if too_far:
            break

    if too_far:
        return group  # Skip group due to too much spread

    # Fill missing values with average of known values
    avg_lat = known['latitude'].mean()
    avg_lon = known['longitude'].mean()

    group['latitude'] = group['latitude'].fillna(avg_lat)
    group['longitude'] = group['longitude'].fillna(avg_lon)
    return group


ufo_loc = ufo_loc.groupby(['city', 'state'], dropna=False).apply(fill_lat_long_within_radius, max_miles=10).reset_index(drop=True)
ufo_loc = ufo_loc.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

ufo_loc['loc'] = ufo_loc[['latitude', 'longitude']].apply(tuple, axis=1)
display(ufo_loc[:20])

C:\Users\nick2\AppData\Local\Temp\ipykernel_22896\3842061845.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ufo_loc = ufo_loc.groupby(['city', 'state'], dropna=False).apply(fill_lat_long_within_radius, max_miles=10).reset_index(drop=True)


,country,state,city,latitude,longitude,loc
0,nan,nan,&ccedil;anakkale (turkey),40.155312,26.414160,"(40.155312, 26.41416)"
1,nan,nan,&iacute;safj&ouml;r&eth;ur (iceland),66.075833,-23.126667,"(66.075833, -23.126667)"
2,nan,nan,&ouml;lmstad (sweden),57.933333,14.366667,"(57.933333, 14.366667)"
3,us,co,1-25 corridor (southbound&#44 65 miles north n...,39.792716,-105.083267,"(39.792716, -105.083267)"
4,ca,bc,100 mile (canada),51.643970,-121.295010,"(51.64397, -121.29501)"
5,ca,bc,100 mile house (canada),51.643970,-121.295010,"(51.64397, -121.29501)"
6,us,ca,29 palms,34.135558,-116.054169,"(34.135558, -116.054169)"
7,us,ca,29 palms,34.135558,-116.054169,"(34.135558, -116.054169)"
8,us,ca,29 palms,34.135558,-116.054169,"(34.135558, -116.054169)"
9,ca,bc,70 mile house (canada),51.303136,-121.395769,"(51.303136, -121.395769)"


I set null lat/long to the average of values with identical city and state columns using haversine distance (suggested by gpt) to only trust that average if all the location data of identical city/states are within 10mi.
Depending what we want to do with this data, dropping ['us_only']=0 may be a good idea. Compared to the bulk of us data, other countries are barely represented.


#### Sub-task 3.  Shape × color co‑occurrence
Extract color keywords from `comments` (red/orange/yellow/green/blue/purple/violet/white/black/silver/gold/pink; handle “-ish” variants). Build a co‑occurrence table with `shape` or your normalized `shape_norm`. Interpret one pairing.


In [ ]:
# Define mapping groups
shape_map = {
    'light': 'light',
    'fireball': 'light',
    'flash': 'light',
    'flare': 'light',

    'unknown': 'nan',
    'other': 'nan',
    'nan': 'nan',   # in case 'nan' is a string

    'circle': 'round',
    'disk': 'round',
    'round': 'round',
    'sphere': 'round',
    'oval': 'round',
    'egg': 'round',
    'dome': 'round',
    'cone': 'round',
    'cylinder': 'round',
    'cigar': 'round',

    'triangle': 'polygon',
    'delta': 'polygon',
    'chevron': 'polygon',
    'rectangle': 'polygon',
    'diamond': 'polygon',
    'pyramid': 'polygon',
    'hexagon': 'polygon',
    'cross': 'polygon',
    'teardrop': 'polygon',
    'crescent': 'polygon',

    'changing': 'changing',
    'changed': 'changing',
    'formation': 'changing',
}

# Apply mapping
ufo_fe['shape_norm'] = ufo_fe['shape'].map(shape_map)
ufo_fe['shape_norm'] = ufo_fe['shape_norm'].fillna('nan')

colors = ["red","orange","yellow","green","blue",
          "purple","violet","white","black",
          "silver","gold","pink"]

# Match color at word boundary, allowing ish/y suffix
pattern = r'\b(' + '|'.join(colors) + r')(?:ish|y)?\b'

def extract_color(text):
    if pd.isna(text):
        return None
    match = re.search(pattern, str(text), flags=re.IGNORECASE)
    if match:
        return match.group(1).lower()  # base color
    return None

ufo_fe['color'] = ufo_fe['comments'].apply(extract_color)


In [10]:
# Cross-tab between color and shape_norm
cooc = pd.crosstab(ufo_fe['color'], ufo_fe['shape_norm'], margins=True, margins_name='Total')
print(cooc)

shape_norm  changing  light   nan  polygon  round  Total
color                                                   
black            101     46   326      845    542   1860
blue              89    882   417      255    833   2476
gold              16     40    39       37     94    226
green             70   1275   437      241    826   2849
orange           420   3068   791      651   2809   7739
pink               8     40    27       38     60    173
purple             3     26    18       16     34     97
red              380   2297  1049      848   1915   6489
silver            45     53   187      103   1115   1503
violet             1      5     2        2      6     16
white            306   2178  1024      924   2229   6661
yellow            75    439   226      146    450   1336
Total           1514  10349  4543     4106  10913  31425


light and round columns contain similar occurances but light is only slightly weighted in colors like red|orange|green over round while round contains more white|yellow and almost all silver data.


#### Sub-task 4. Spatio‑temporal spikes (anomaly cues)
For each geocell, create a daily count series; compute z‑scores within cell; list top‑10 spikes across all cells. Discuss plausible causes.


## (40 points) Part 2: Association Rules in Movie Rating Behaviors

For the second part of this homework, we're going to examine movies using our understanding of association rules, to find movies that "go together". For this part, you will implement the apriori algorithm, and apply it to a movie rating dataset. We'll use the [MovieLens](https://grouplens.org/datasets/movielens/) dataset.

First, run the next cell to load the dataset we are going to use.

In [11]:
import urllib3
import zipfile

http = urllib3.PoolManager()
req = http.request("GET", "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip", preload_content=False)

with open("movie.zip", 'wb') as out:
  while True:
    data = req.read(4096)
    if not data:
      break
    out.write(data)
req.release_conn()

zFile = zipfile.ZipFile("movie.zip", "r")
for fileM in zFile.namelist():
  zFile.extract(fileM)

In [20]:
# !ls ml-latest-small/

In this dataset, there are four columns: `userId` is the integer ids of users, `movieId` is the integer ids of movies, `rating` is the rate of the user gives to the movie, and `timestamp` which we do not use here. Each row denotes that the user of given `userId` rated the movie of the given `movieId`. We are going to treat each user as a "basket", so you will need to collect all the movies that have been rated by a single user as a basket.

Now, you need to implement the apriori algorithm and apply it to this dataset to find association rules of user rating behaviors where:

1. Define `rating` >= 3 is "like" (that is, only consider movie ratings of 3 or higher in your baskets; you may ignore all others)
2. `minsup` == 40 (out of 600 users/baskets); we may adjust this based on the discussion on Canvas
3. `minconf` == to be determined by a discussion on Canvas. You may try several different choices, but we will converge on a good choice for everyone for the final submission.

We know there are many existing implementations of apriori online (check github for some good starting points). You are welcome to read existing codebases and let that inform your approach. Do not copy-paste any existing code. We want your code to have sufficient comments to explain your steps, to show us that you really know what you are doing. Furthermore, you should add print statements to print out the intermediate steps of your method -- e.g., the size of the candidate set at each step of the method, the size of the filtered set, and any other important information you think will highlight the method.

To help get you started, we can load the ratings with the following code snippet:

In [22]:
import pandas as pd
# read user ratings
allRatings = pd.read_csv("ml-latest-small/ratings.csv")
allRatings.columns

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')

In [ ]:
likes_df = (
    allRatings[allRatings['rating'] >= 3.0]
    .groupby('userId')['movieId']
    .apply(list)
    .reset_index()
    .rename(columns={'movieId': 'likes'})
)
display(likes_df.head())

,userId,likes
0,1,"[1, 3, 6, 47, 50, 70, 101, 110, 151, 157, 163,..."
1,2,"[318, 333, 1704, 3578, 6874, 8798, 46970, 4851..."
2,3,"[849, 1275, 1371, 1587, 2288, 2851, 3024, 3703..."
3,4,"[21, 45, 52, 58, 106, 125, 162, 171, 176, 215,..."
4,5,"[1, 21, 34, 36, 39, 50, 58, 110, 150, 153, 232..."


int

### (15pts) Step 1: Implement Apriori Algorithm
In this section, you need to implement the Apriori algorithm, we will check the correctness of your code and we encourage efficient implementation.

In [50]:
import itertools

def candid(F, transactions, minsup, k):
    Fk = dict()
    if k == 1: # Size 1 baskets
        for _, t in transactions.iterrows():
            for i in t['likes']: # create value counts for each movie
                if i in Fk.keys(): Fk[i] += 1
                else: Fk[i] = 1
        # drop movies that don't meet minsup
        Fk = {i: sup for i, sup in Fk.items() if sup >= minsup}
        return Fk
    if k == 2: # Size 2 baskets
        # itertools will create all possible pairs of frequent 1-itemsets
        # but this wont be effective for larger baskets since we want to only add possible candidates by appending 1 item from other candidates at a time
        candidates = list(itertools.combinations(F[k-2].keys(), 2))
    else:
        # --- General case: extend (k-1)-itemsets by one new item ---
        S = set()
        for key in F[k-2].keys():
            S.update(key)
        candidates = set()
        for key in F[k-2].keys():
            base = key
            for item in S:
                if item not in base:
                    new_cand = tuple(sorted(base + (item,)))
                    candidates.add(new_cand)

    # Count support
    if k >= 2:
        for _, t in transactions.iterrows():
            likes = set(t['likes'])
            for c in candidates:
                if set(c).issubset(likes):
                    Fk[c] = Fk.get(c, 0) + 1
        return {c: sup for c, sup in Fk.items() if sup >= minsup}
    
    
def Apriori(transactions, minsup):
    k = 1
    F = []
    F.append(candid(F, transactions, minsup, k))
    print(f'Found {len(F[k - 1])} frequent {k}-itemsets')
    while F[k - 1]:
        k += 1
        F.append(candid(F, transactions, minsup, k))
        print(f'Found {len(F[k - 1])} frequent {k}-itemsets')
    return F

Ab4I = Apriori(likes_df, minsup=60)

Found 259 frequent 1-itemsets
Found 2856 frequent 2-itemsets
Found 7337 frequent 3-itemsets
Found 7497 frequent 4-itemsets
Found 3407 frequent 5-itemsets
Found 687 frequent 6-itemsets
Found 46 frequent 7-itemsets
Found 1 frequent 8-itemsets
Found 0 frequent 9-itemsets


In [51]:
print(Ab4I)

[{1: 199, 6: 99, 47: 191, 50: 197, 110: 218, 223: 94, 231: 88, 235: 62, 260: 234, 296: 287, 316: 119, 349: 102, 356: 315, 367: 122, 457: 182, 480: 216, 500: 121, 527: 205, 590: 152, 592: 172, 593: 263, 608: 171, 648: 145, 733: 112, 736: 93, 780: 164, 919: 83, 923: 62, 1073: 112, 1080: 78, 1089: 124, 1097: 105, 1136: 126, 1196: 199, 1197: 135, 1198: 191, 1206: 109, 1208: 101, 1210: 185, 1213: 122, 1214: 130, 1220: 78, 1222: 96, 1240: 120, 1258: 102, 1265: 133, 1270: 159, 1278: 63, 1291: 135, 1517: 84, 1573: 60, 1580: 139, 1617: 92, 1625: 63, 1676: 62, 1732: 95, 2000: 71, 2012: 71, 2028: 179, 2115: 97, 2174: 78, 2291: 74, 2329: 127, 2502: 88, 2542: 65, 2571: 257, 2628: 97, 2692: 72, 2700: 69, 2716: 109, 2797: 83, 2858: 186, 2916: 81, 2959: 206, 2987: 85, 2997: 91, 3052: 66, 3147: 106, 3578: 153, 3793: 121, 318: 313, 1704: 136, 6874: 123, 48516: 103, 58559: 144, 68157: 85, 74458: 60, 79132: 129, 91529: 71, 99114: 65, 109487: 70, 21: 77, 357: 86, 368: 67, 588: 169, 595: 137, 904: 81, 912: 

### (5pts) Step 2: Print Your Association Rules

Next you should print your final association rules in the following format:

**movie_name_1, movie_name_2, ... -->
movie_name_k**

where the movie names can be fetched by joining the movieId with the file `movies.csv`. For example, one rule that you might find is:

**Matrix, The (1999),  Star Wars: Episode V - The Empire Strikes Back (1980),  Star Wars: Episode IV - A New Hope (1977),  ->
Star Wars: Episode VI - Return of the Jedi (1983)**

In [52]:
# Store the top 25 scores (v) per itemset length in 'top'
top = {i: [] for i in range(1, 9)}  # 1 to 8

for d in Ab4I:
    for k, v in d.items():
        klen = len(k) if isinstance(k, tuple) else 1
        top[klen].append((k, v))

# Keep only the top 25 by score for each length
for klen in top:
    top[klen] = sorted(top[klen], key=lambda x: x[1], reverse=True)[:25]

print(top)

{1: [(356, 315), (318, 313), (296, 287), (593, 263), (2571, 257), (260, 234), (110, 218), (480, 216), (589, 210), (2959, 206), (527, 205), (1, 199), (1196, 199), (50, 197), (47, 191), (1198, 191), (2858, 186), (1210, 185), (150, 185), (4993, 183), (457, 182), (2028, 179), (858, 178), (5952, 174), (7153, 174)], 2: [((356, 318), 221), ((296, 318), 209), ((296, 356), 207), ((296, 593), 188), ((593, 318), 187), ((356, 593), 183), ((356, 480), 177), ((260, 1196), 176), ((356, 2571), 173), ((2571, 318), 166), ((110, 356), 165), ((260, 1210), 161), ((260, 2571), 161), ((296, 2571), 160), ((47, 296), 158), ((2571, 2959), 158), ((1196, 2571), 157), ((527, 318), 157), ((50, 318), 156), ((50, 296), 155), ((110, 296), 155), ((356, 527), 155), ((110, 318), 155), ((4993, 5952), 155), ((4993, 7153), 155)], 3: [((296, 318, 356), 161), ((296, 318, 593), 151), ((296, 356, 593), 145), ((260, 1196, 2571), 144), ((4993, 5952, 7153), 144), ((318, 356, 593), 143), ((260, 1196, 1210), 142), ((296, 356, 2571),

### (10pts) Step 3: Implement Random Sampling

We discussed in class a method to randomly sample baskets to avoid the overhead of reading the entire set of baskets (which in practice, could amount to billions of baskets). For this part, you should implement such a random sampling approach that takes a special parameter **alpha** that controls the size of the sample: e.g., alpha = 0.10 means to sample 10% of the baskets (our users, in this case).

Vary **alpha** and report the number of frequent itemsets you find and how this compares to the number of frequent itemsets in the entire dataset. What do you discover?


In [16]:
# your code here

*your discussion here*

### (10pts) Step 4: Check for False Positives

Next you should verify that the candidate pairs you discover by random sampling are truly frequent by comparing to the itemsets you discover over the entire dataset.

For this part, consider another parameter **minsup_sample** that relaxes the minimum support threshold. For example if we want minsup = 1/100 for whole dataset, then try minsup_sample = 1/125 for the sample. This will help catch truly frequent itemsets.

Vary **minsup_sample** and report the number of frequent itemsets you find and the number of false positives you find. What do you discover?


In [17]:
# your code here

*your discussion here*

## (5 points) Part 3: Spark-based Association Rules

So far, we have been working with a fairly small dataset. For this last question, you should use the much larger **Movies 10M** dataset: https://files.grouplens.org/datasets/movielens/ml-10m.zip

First, we need to load this larger dataset:

In [18]:
import urllib3
import zipfile

http = urllib3.PoolManager()
req = http.request("GET", "https://files.grouplens.org/datasets/movielens/ml-10m.zip", preload_content=False)

with open("movie.zip", 'wb') as out:
  while True:
    data = req.read(4096)
    if not data:
      break
    out.write(data)
req.release_conn()

zFile = zipfile.ZipFile("movie.zip", "r")
for fileM in zFile.namelist():
  zFile.extract(fileM)

Now, see if you can write a Spark-based implementaiton of Apriori. You'll see that the Spark library MLib does contain a "Frequent Pattern Mining" implementation. You may not use this! Good luck!

In [19]:
# your code here